# Self-Organizing Map (SOM): Lab 04
**Name / Reg No:** Urva Ishfaq/FA23-BAI-051

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## Task 1: Data and Weight Setup

In [ ]:
# 1. 200 random colors (R, G, B in [0, 1])
data = np.random.rand(200, 3)

# 2. 10 x 10 grid of neurons, each with a random 3-value weight vector
grid_size = 10
weights = np.random.rand(grid_size, grid_size, 3)
initial_weights = weights.copy()

# 3. Display the initial weight grid
plt.imshow(weights)
plt.title("Initial Weight Grid")
plt.axis("off")
plt.show()

# 4. Shape, min and max
print("Data    -> shape:", data.shape, "| min: %.4f | max: %.4f" % (data.min(), data.max()))
print("Weights -> shape:", weights.shape, "| min: %.4f | max: %.4f" % (weights.min(), weights.max()))

## Task 2: Distance Calculation

In [ ]:
# 1. Euclidean distance
def euclidean_distance(a, b):
    return np.sqrt(np.sum((np.asarray(a) - np.asarray(b)) ** 2))

print("Test:", euclidean_distance([0, 0, 0], [1, 1, 1]))  # sqrt(3)

# 2. Vectorized distance from one random sample to every neuron (10 x 10)
x = data[np.random.randint(len(data))]
dist = np.sqrt(np.sum((weights - x) ** 2, axis=2))
print("Distance array shape:", dist.shape)

# 3. Heatmap with colorbar
plt.imshow(dist, cmap="viridis")
plt.colorbar(label="Distance")
plt.title("Distance from Sample to Every Neuron")
plt.show()

# 4. Smallest distance and its location
r, c = np.unravel_index(np.argmin(dist), dist.shape)
print("Smallest distance: %.4f at row %d, column %d" % (dist[r, c], r, c))

## Task 3: Finding the Winner (BMU)

In [ ]:
# 1. Best Matching Unit
def find_bmu(weights, x):
    d = np.sqrt(np.sum((weights - x) ** 2, axis=2))
    idx = np.unravel_index(np.argmin(d), d.shape)
    return (int(idx[0]), int(idx[1]))

# 2. Five different samples
print(f"{'Sample':<8}{'BMU (row, col)':<18}{'Distance'}")
for i in np.random.choice(len(data), 5, replace=False):
    b = find_bmu(weights, data[i])
    print(f"{i:<8}{str(b):<18}{euclidean_distance(weights[b], data[i]):.4f}")

# 3. Weight grid with BMU of one sample marked in red
sample = data[0]
bmu = find_bmu(weights, sample)
plt.imshow(weights)
plt.scatter(bmu[1], bmu[0], c="red", s=150, marker="x")   # x = column, y = row
plt.title(f"BMU of sample 0 at {bmu}")
plt.show()

# 4. Hit map for all 200 samples
hits = np.zeros((grid_size, grid_size), dtype=int)
for s in data:
    r, c = find_bmu(weights, s)
    hits[r, c] += 1
plt.imshow(hits, cmap="hot")
plt.colorbar(label="Number of samples")
plt.title("Hit Map (untrained SOM)")
plt.show()
print("Total samples counted:", hits.sum())

## Task 4: Neighborhood Function

In [ ]:
# 1. Gaussian neighborhood
def neighborhood(grid_size, bmu, radius):
    rows, cols = np.indices((grid_size, grid_size))
    d2 = (rows - bmu[0]) ** 2 + (cols - bmu[1]) ** 2
    return np.exp(-d2 / (2 * radius ** 2))

# 2. BMU (5,5), radius 1, 3, 5
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, rad in zip(axes, [1, 3, 5]):
    im = ax.imshow(neighborhood(10, (5, 5), rad), cmap="viridis")
    ax.set_title(f"Gaussian, radius = {rad}")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

# 3. Bubble neighborhood
def neighborhood_bubble(grid_size, bmu, radius):
    rows, cols = np.indices((grid_size, grid_size))
    dist = np.sqrt((rows - bmu[0]) ** 2 + (cols - bmu[1]) ** 2)
    h = np.exp(-dist ** 2 / (2 * radius ** 2))
    h[dist > radius] = 0
    return h

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for j, rad in enumerate([1, 3, 5]):
    axes[0, j].imshow(neighborhood(10, (5, 5), rad)); axes[0, j].set_title(f"Gaussian r={rad}")
    axes[1, j].imshow(neighborhood_bubble(10, (5, 5), rad)); axes[1, j].set_title(f"Bubble r={rad}")
plt.tight_layout()
plt.show()

# 4. h at BMU and 3 steps away (radius 2)
h = neighborhood(10, (5, 5), 2)
print("h at BMU (5,5):            %.4f" % h[5, 5])
print("h at 3 steps away (5,8):   %.4f" % h[5, 8])

## Task 5: One Weight Update Step

In [ ]:
# 1. One SOM update step
def update_weights(weights, x, bmu, learning_rate, radius):
    h = neighborhood(weights.shape[0], bmu, radius)
    new_w = weights + learning_rate * h[:, :, None] * (x - weights)
    return new_w

# 2. Apply once, compare before / after
x = data[0]
bmu = find_bmu(weights, x)
before = weights.copy()
dist_before = euclidean_distance(before[bmu], x)
after = update_weights(weights, x, bmu, learning_rate=0.5, radius=3)
dist_after = euclidean_distance(after[bmu], x)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(before); axes[0].set_title("Before update")
axes[1].imshow(after);  axes[1].set_title("After update")
plt.show()

# 3. Total absolute change
print("Total absolute change in weights: %.4f" % np.sum(np.abs(after - before)))

# 4. BMU-sample distance before / after
print("Distance before: %.4f" % dist_before)
print("Distance after:  %.4f" % dist_after)
if dist_after < dist_before:
    print("Confirmed: the distance decreased.")
else:
    print("The distance did NOT decrease.")

## Task 6: Training Loop

In [ ]:
# 1 & 2. Full training with exponential decay; lr and radius stored every 100 iterations
def train_som(data, grid_size, iterations, lr_start, radius_start, history=None, seed=0):
    rng = np.random.default_rng(seed)
    w = rng.random((grid_size, grid_size, 3))
    tau = iterations / np.log(max(radius_start, 1.01))   # radius decays to ~1
    for t in range(iterations):
        lr = lr_start * np.exp(-t / iterations)
        radius = radius_start * np.exp(-t / tau)
        x = data[rng.integers(len(data))]
        bmu = find_bmu(w, x)
        w = update_weights(w, x, bmu, lr, radius)
        if history is not None and t % 100 == 0:
            history["lr"].append(lr)
            history["radius"].append(radius)
    return w

hist6 = {"lr": [], "radius": []}
initial_w6 = np.random.default_rng(0).random((10, 10, 3))   # same init as train_som(seed=0)
trained = train_som(data, 10, 3000, lr_start=0.5, radius_start=5, history=hist6)

# 3. Plot both lists
steps = np.arange(0, 3000, 100)
plt.plot(steps, hist6["lr"], label="Learning rate")
plt.plot(steps, hist6["radius"], label="Radius")
plt.xlabel("Iteration"); plt.ylabel("Value"); plt.title("Decay of Learning Rate and Radius")
plt.legend(); plt.show()

# 4. Initial vs trained, saved to file
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(initial_w6); axes[0].set_title("Initial")
axes[1].imshow(trained);    axes[1].set_title("Trained")
plt.savefig("task6_result.png", dpi=150, bbox_inches="tight")
plt.show()

## Task 7: Measuring Training Quality

In [ ]:
# 1. Quantization error
def quantization_error(data, weights):
    flat = weights.reshape(-1, 3)
    d = np.sqrt(((data[:, None, :] - flat[None, :, :]) ** 2).sum(axis=2))
    return d.min(axis=1).mean()

# 2. Training function that also records error every 250 iterations
def train_som(data, grid_size, iterations, lr_start, radius_start, history=None, seed=0):
    rng = np.random.default_rng(seed)
    w = rng.random((grid_size, grid_size, 3))
    tau = iterations / np.log(max(radius_start, 1.01))
    for t in range(iterations):
        lr = lr_start * np.exp(-t / iterations)
        radius = radius_start * np.exp(-t / tau)
        x = data[rng.integers(len(data))]
        bmu = find_bmu(w, x)
        w = update_weights(w, x, bmu, lr, radius)
        if history is not None:
            if t % 100 == 0:
                history["lr"].append(lr)
                history["radius"].append(radius)
            if t % 250 == 0:
                history["error"].append(quantization_error(data, w))
                history["error_iter"].append(t)
    return w

hist7 = {"lr": [], "radius": [], "error": [], "error_iter": []}
trained7 = train_som(data, 10, 3000, 0.5, 5, history=hist7)

# 3. Error vs iteration
plt.plot(hist7["error_iter"], hist7["error"], marker="o")
plt.xlabel("Iteration"); plt.ylabel("Quantization error")
plt.title("Quantization Error During Training")
plt.grid(True); plt.show()

# 4. First, last, % reduction
first, last = hist7["error"][0], hist7["error"][-1]
print("First error: %.4f" % first)
print("Last error:  %.4f" % last)
print("Reduction:   %.2f%%" % ((first - last) / first * 100))

## Task 8: Parameter Experiments

In [ ]:
results = []   # (setting name, value, final error)

# 1. Iterations
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, it in zip(axes, [10, 500, 5000]):
    w = train_som(data, 10, it, 0.5, 5)
    err = quantization_error(data, w)
    results.append(("Iterations", it, err))
    ax.imshow(w); ax.set_title(f"Iterations = {it}")
plt.tight_layout(); plt.show()

# 2. Grid sizes
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, g in zip(axes, [5, 10, 20]):
    w = train_som(data, g, 2000, 0.5, g / 2)
    err = quantization_error(data, w)
    results.append(("Grid size", g, err))
    ax.imshow(w); ax.set_title(f"Grid = {g} x {g}")
plt.tight_layout(); plt.show()

# 3. Learning rates
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, lr in zip(axes, [0.01, 0.5, 0.9]):
    w = train_som(data, 10, 2000, lr, 5)
    err = quantization_error(data, w)
    results.append(("Learning rate", lr, err))
    ax.imshow(w); ax.set_title(f"Learning rate = {lr}")
plt.tight_layout(); plt.show()

# 4. Summary table
print(f"{'Setting':<16}{'Value':<10}{'Final error'}")
print("-" * 38)
for name, val, err in results:
    print(f"{name:<16}{val:<10}{err:.4f}")

## Task 9: Mini Project with Your Own Data

In [ ]:
# 1. 100 samples near red, green, blue (noise, clipped to [0, 1])
base = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=float)
rng = np.random.default_rng(1)
mini = base[rng.integers(0, 3, 100)] + rng.normal(0, 0.1, (100, 3))
mini = np.clip(mini, 0, 1)
print("Mini data shape:", mini.shape, "| min %.3f | max %.3f" % (mini.min(), mini.max()))

# 2. Train a 10 x 10 SOM and show it
w9 = train_som(mini, 10, 3000, 0.5, 5)
plt.imshow(w9); plt.title("Trained SOM (red / green / blue data)"); plt.axis("off"); plt.show()

# 3. Group samples by BMU
groups = {}
for i, s in enumerate(mini):
    groups.setdefault(find_bmu(w9, s), []).append(i)

# 4. Count of used neurons + hit map
print("Neurons with at least one sample:", len(groups), "out of 100")
hits9 = np.zeros((10, 10), dtype=int)
for (r, c), idxs in groups.items():
    hits9[r, c] = len(idxs)
plt.imshow(hits9, cmap="hot"); plt.colorbar(label="Samples")
plt.title("Hit Map: separate groups should appear for R, G, B"); plt.show()